In [ ]:
import pandas as pd
import folium
import plotly.express as px

from folium.plugins import HeatMap
from folium.plugins import MarkerCluster

In [ ]:
earthquakes = pd.read_csv(
    "data/processed/earthquakes_processed.csv"
)

earthquakes.head()

In [ ]:
earthquakes.info()

In [ ]:
earthquakes = earthquakes.dropna(
    subset=[
        "Latitude",
        "Longitude",
        "Magnitude"
    ]
)

print(earthquakes.shape)

In [ ]:
earthquakes["Time"] = pd.to_datetime(
    earthquakes["Time"],
    format="mixed",
    utc=True,
    errors="coerce"
)

In [ ]:
world_map = folium.Map(
    location=[0, 0],
    zoom_start=2,
    tiles="CartoDB positron"
)

world_map

In [ ]:
for _, row in earthquakes.iterrows():

    folium.CircleMarker(
        location=[
            row["Latitude"],
            row["Longitude"]
        ],

        radius=max(row["Magnitude"] * 2, 2),

        popup=(
            f"Location: {row['Location']}<br>"
            f"Magnitude: {row['Magnitude']}<br>"
            f"Depth: {row['Depth_km']} km"
        ),

        color="red",

        fill=True,

        fill_opacity=0.6

    ).add_to(world_map)

world_map

In [ ]:
world_map.save(
    "outputs/maps/earthquake_map.html"
)

print("Map saved successfully.")

In [ ]:
cluster_map = folium.Map(
    location=[0, 0],
    zoom_start=2
)

marker_cluster = MarkerCluster()

for _, row in earthquakes.iterrows():

    folium.Marker(
        location=[
            row["Latitude"],
            row["Longitude"]
        ],

        popup=f"{row['Location']}<br>Magnitude: {row['Magnitude']}"
    ).add_to(marker_cluster)

marker_cluster.add_to(cluster_map)

cluster_map

In [ ]:
cluster_map.save(
    "outputs/maps/cluster_map.html"
)

In [ ]:
heat_data = earthquakes[
    [
        "Latitude",
        "Longitude",
        "Magnitude"
    ]
].dropna().values.tolist()

heat_map = folium.Map(
    location=[0, 0],
    zoom_start=2
)

HeatMap(
    heat_data,
    radius=10
).add_to(heat_map)

heat_map

In [ ]:
heat_map.save(
    "outputs/maps/heat_map.html"
)

In [ ]:
plot_df = earthquakes.copy()

plot_df["Marker_Size"] = (
    plot_df["Magnitude"].clip(lower=0) + 1
)

fig = px.scatter_geo(
    plot_df,
    lat="Latitude",
    lon="Longitude",
    color="Magnitude",
    size="Marker_Size",
    hover_name="Location",
    hover_data=[
        "Depth_km",
        "Magnitude"
    ],
    projection="natural earth",
    title="Global Earthquake Distribution"
)

fig.show()

In [ ]:
fig.write_html(
    "outputs/figures/global_earthquake_distribution.html"
)

print("Plotly figure saved.")